# Customer Churn Prediction

In [ ]:
#main.py - import libraries
import pandas as pd
import numpy as np
from sklearn.tree import DecisionTreeClassifier, export_text
from sklearn.model_selection import GridSearchCV
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, accuracy_score, precision_score, recall_score, f1_score


In [592]:
#main.py - read in data
test = pd.read_csv("/Users/grace/Downloads/mat311/final_project/customer_churn_prediction/data/raw/test.csv")
train = pd.read_csv("/Users/grace/Downloads/mat311/final_project/customer_churn_prediction/data/raw/train.csv")

test.head()

,CustomerID,Age,Gender,Tenure,Usage Frequency,Support Calls,Payment Delay,Subscription Type,Contract Length,Total Spend,Last Interaction,Last Due Date,Last Payment Date,Customer Status
0,146773,43,Male,25.0,28,9,18,Standard,Monthly,809.00,7.0,06-30,07-18,active
1,21394,60,Male,NaN,21,8,20,Standard,Annual,890.00,28.0,06-01,06-21,active
2,411099,29,Male,58.0,15,1,1,Standard,Quarterly,981.25,20.0,06-26,06-27,active
3,239666,35,Male,NaN,28,5,5,Basic,Quarterly,321.64,22.0,06-05,06-10,active
4,35032,33,Female,NaN,12,6,13,Premium,Monthly,437.00,NaN,06-26,07-09,active


In [593]:
# preprocessing.py

def fill_na_with_mode(df: pd.DataFrame, columns: list) -> pd.DataFrame:
    """Fill missing values in the specified columns of the DataFrame with the mode."""
    for col in columns:
        mode_value = df[col].mode()[0]
        df[col] = df[col].fillna(mode_value)
    return df

def clean_dataset_simple(df: pd.DataFrame) -> pd.DataFrame:
    """Clean the dataset by dropping rows with missing values."""
    return df.dropna().copy()

def fill_cols_na_with_0s(df: pd.DataFrame, columns: list) -> pd.DataFrame:
    """Fill missing values in the specified columns of the DataFrame with 0s."""
    for col in columns:
        df[col] = pd.to_numeric(df[col], errors='coerce').fillna(0)
    return df

def one_hot_encode(df: pd.DataFrame, columns: list) -> pd.DataFrame:
    """One-hot encode specified categorical columns."""
    return pd.get_dummies(df, columns=columns, drop_first=True)

def convert_data_types(df: pd.DataFrame, column_types: dict) -> pd.DataFrame:
    """Convert columns to specified data types."""
    for column, dtype in column_types.items():
        df[column] = df[column].astype(dtype)
    return df



In [594]:
#train knn model
def train_knn_model(X_train: pd.DataFrame, y_train: pd.Series, ) -> KNeighborsClassifier:
    """Train and return a 3-NN classifier."""
    model = KNeighborsClassifier(n_neighbors=3)
    model.fit(X_train, y_train)
    return model

In [595]:
def preprocess_test(df):
    df_clean = df.copy()

    df_clean = df_clean.drop(['Last Due Date', 'Last Payment Date'], axis=1) #drop date format
    df_clean = fill_cols_na_with_0s(df_clean, columns=['Support Calls'])
    df_clean = fill_na_with_mode(df_clean, columns=['Tenure','Last Interaction', 'Payment Delay'])

    # Remove Customer Status to avoid data leakage
    df_clean = df_clean.drop(columns=['Customer Status'], errors='ignore')

    # One-hot encode / map binary categorical variables
    df_clean['Gender'] = df_clean['Gender'].map({'Male': 0, 'Female': 1})
    df_clean = one_hot_encode(df_clean, columns=['Subscription Type', 'Contract Length'])
    
    # Convert data types
    df_clean = convert_data_types(df_clean, column_types={'Support Calls': 'int'})
    

    return df_clean

In [608]:
def preprocess_train(df, target_col='Churn'):
    df_clean = df.copy()
    
    # Store target separately
    y = df_clean[target_col]
    X = df_clean.drop(columns=[target_col])
    
    # Apply preprocessing to X
    X = X.drop(['Last Due Date', 'Last Payment Date'], axis=1, errors='ignore') #drop date format
    X = fill_cols_na_with_0s(X, columns=['Support Calls'])
    X = fill_na_with_mode(X, columns=['Tenure','Last Interaction', 'Payment Delay'])
    
    # Remove Customer Status to avoid data leakage
    X = X.drop(columns=['Customer Status'], errors='ignore')
    
    # One-hot encode / map binary categorical variables
    X['Gender'] = X['Gender'].map({'Male': 0, 'Female': 1})
    print(X[['Gender']].value_counts())
    X = one_hot_encode(X, columns=['Subscription Type', 'Contract Length'])
    
    # Convert data types
    X = convert_data_types(X, column_types={'Support Calls': 'int'})
    
    # Get indices that remain after al preprocessing
    # (some preprocessing might drop rows)
    final_indices = X.index
    
    # Return aligned X and y
    return X, y.loc[final_indices]

X, y = preprocess_train(train)
assert X.shape[0] == y.shape[0], "Mismatch in number of samples between X and y"
assert X.isnull().sum().sum() == 0, "There are still missing values in X"
print(f"Aligned data - X: {X.shape}, y: {y.shape}")

Gender
0         181117
1         122007
Name: count, dtype: int64
Aligned data - X: (303124, 13), y: (303124,)


In [597]:
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.3, random_state=42)



In [598]:
knnModel = train_knn_model(X_train, y_train)

In [599]:
test_clean = preprocess_test(test)


# Align test columns with training columns
missing_cols = set(X_train.columns) - set(test_clean.columns)
for col in missing_cols:
    test_clean[col] = 0  # add missing columns with 0

extra_cols = set(test_clean.columns) - set(X_train.columns)
test_clean = test_clean.drop(columns=list(extra_cols))  # drop extra columns

# Reorder test columns to match training
test_clean = test_clean[X_train.columns]

In [600]:
test_clean.head()

,CustomerID,Age,Gender,Tenure,Usage Frequency,Support Calls,Payment Delay,Total Spend,Last Interaction,Subscription Type_Premium,Subscription Type_Standard,Contract Length_Monthly,Contract Length_Quarterly
0,146773,43,0,25.0,28,9,18,809.00,7.0,False,True,True,False
1,21394,60,0,55.0,21,8,20,890.00,28.0,False,True,False,False
2,411099,29,0,58.0,15,1,1,981.25,20.0,False,True,False,True
3,239666,35,0,55.0,28,5,5,321.64,22.0,False,False,False,True
4,35032,33,1,55.0,12,6,13,437.00,7.0,True,False,True,False


In [601]:
X.head()

,CustomerID,Age,Gender,Tenure,Usage Frequency,Support Calls,Payment Delay,Total Spend,Last Interaction,Subscription Type_Premium,Subscription Type_Standard,Contract Length_Monthly,Contract Length_Quarterly
0,321685,47,0,52.0,9,4,10.0,738.64,7.0,False,False,False,True
1,407818,45,1,13.0,16,1,0.0,822.72,7.0,False,True,False,False
2,250306,33,1,56.0,29,6,0.0,462.62,29.0,False,False,False,True
3,308397,40,0,13.0,25,3,4.0,516.91,1.0,False,False,False,True
4,298181,38,0,18.0,16,2,2.0,896.28,7.0,True,False,False,False


In [602]:
assert test_clean.isnull().sum().sum() == 0, "There are still missing values in test_clean"
print(test_clean.shape)
print(X.shape)

assert test_clean.shape[1] == X.shape[1], "Mismatch in number of features between test_clean and training data X"


(133776, 13)
(303124, 13)


In [603]:
#validation predictions
y_pred = knnModel.predict(X_val)
y_proba = knnModel.predict_proba(X_val)

In [604]:
# confusion matrix on training data
conf_knn = confusion_matrix(y_val, y_pred)
conf_knn 

array([[62430,  1269],
       [ 1534, 25705]])

In [605]:
#final test predictions
y_pred_test = knnModel.predict(test_clean)
y_proba_test = knnModel.predict_proba(test_clean)

y_proba_test = y_proba_test[:,1]

assert test_clean['CustomerID'].equals(test['CustomerID'])
test_clean.size


1739088

In [ ]:


# Initialize the Decision Tree classifier
dt_model = DecisionTreeClassifier(
    criterion='gini',      
    max_depth=None,        
    random_state=42
)

# Fit the model on training data
dt_model.fit(X_train, y_train)

# Predict on validation set
y_val_pred = dt_model.predict(X_val)

# Evaluate performance
print("Validation Accuracy:", accuracy_score(y_val, y_val_pred))
print("\nConfusion Matrix:\n", confusion_matrix(y_val, y_val_pred))


# Predict on test set
y_test_pred = dt_model.predict(X_test)  # assuming X_test is preprocessed

Validation Accuracy: 0.9938859442697222

Confusion Matrix:
 [[63444   255]
 [  301 26938]]


ValueError: The feature names should match those that were passed during fit.
Feature names unseen at fit time:
- Customer Status_inactive
- Gender_Male
Feature names seen at fit time, yet now missing:
- Gender


In [ ]:

"""DONT FORGET TO UPDATE SUBMISSION FILE NAME"""
submission = pd.DataFrame({
    'CustomerID': test['CustomerID'], 
    'Churn': y_test_pred
})
submission.to_csv("/Users/grace/Downloads/mat311/final_project/customer_churn_prediction/data/processed/knn_predictions_03.csv", index=False)

In [ ]:
print(f"Indices match: {all(test_clean['CustomerID'] == test['CustomerID'])}")


Indices match: True


In [ ]:
print("=== COMPREHENSIVE CUSTOMERID INVESTIGATION ===")
print(f"Original test shape: {test.shape}") 
print(f"Preprocessed test_clean shape: {test_clean.shape}")

# Check columns in both datasets
print(f"\nOriginal test columns: {test.columns.tolist()}")
print(f"Preprocessed test_clean columns: {test_clean.columns.tolist()}")

# Check if CustomerID exists as a column
print(f"\n'CustomerID' in test columns: {'CustomerID' in test.columns}")
print(f"'CustomerID' in test_clean columns: {'CustomerID' in test_clean.columns}")

# Check index information
print(f"\nTest index range: {test.index[0]} to {test.index[-1]}")
print(f"Test_clean index range: {test_clean.index[0]} to {test_clean.index[-1]}")
print(f"Index types match: {type(test.index) == type(test_clean.index)}")

# Sample values comparison
print(f"\nFirst 5 test indices: {test.index[:5].tolist()}")
print(f"First 5 test_clean indices: {test_clean.index[:5].tolist()}")

if 'CustomerID' in test.columns:
    print(f"First 5 test CustomerID values: {test['CustomerID'].head().tolist()}")
if 'CustomerID' in test_clean.columns:
    print(f"First 5 test_clean CustomerID values: {test_clean['CustomerID'].head().tolist()}")

=== COMPREHENSIVE CUSTOMERID INVESTIGATION ===
Original test shape: (133776, 14)
Preprocessed test_clean shape: (133776, 13)

Original test columns: ['CustomerID', 'Age', 'Gender', 'Tenure', 'Usage Frequency', 'Support Calls', 'Payment Delay', 'Subscription Type', 'Contract Length', 'Total Spend', 'Last Interaction', 'Last Due Date', 'Last Payment Date', 'Customer Status']
Preprocessed test_clean columns: ['CustomerID', 'Age', 'Gender', 'Tenure', 'Usage Frequency', 'Support Calls', 'Payment Delay', 'Total Spend', 'Last Interaction', 'Subscription Type_Premium', 'Subscription Type_Standard', 'Contract Length_Monthly', 'Contract Length_Quarterly']

'CustomerID' in test columns: True
'CustomerID' in test_clean columns: True

Test index range: 0 to 133775
Test_clean index range: 0 to 133775
Index types match: True

First 5 test indices: [0, 1, 2, 3, 4]
First 5 test_clean indices: [0, 1, 2, 3, 4]
First 5 test CustomerID values: [146773, 21394, 411099, 239666, 35032]
First 5 test_clean Custo

In [ ]:
# Check if your test data has the right CustomerIDs
print("Are we using the correct test file?")
print(f"Number of unique CustomerIDs: {test_clean['CustomerID'].nunique()}")
print(f"Total rows: {len(test_clean)}")

# Sample of CustomerIDs
print(f"First 10 CustomerIDs: {test_clean['CustomerID'].head(10).tolist()}")
print(f"Last 10 CustomerIDs: {test_clean['CustomerID'].tail(10).tolist()}")

# Sample of CustomerIDs
print(f"First 10 CustomerIDs: {test['CustomerID'].head(10).tolist()}")
print(f"Last 10 CustomerIDs: {test['CustomerID'].tail(10).tolist()}")

Are we using the correct test file?
Number of unique CustomerIDs: 133776
Total rows: 133776
First 10 CustomerIDs: [146773, 21394, 411099, 239666, 35032, 248710, 199062, 43257, 47393, 95561]
Last 10 CustomerIDs: [236902, 57360, 155542, 148289, 154501, 223515, 445376, 239361, 135584, 309112]
First 10 CustomerIDs: [146773, 21394, 411099, 239666, 35032, 248710, 199062, 43257, 47393, 95561]
Last 10 CustomerIDs: [236902, 57360, 155542, 148289, 154501, 223515, 445376, 239361, 135584, 309112]
